# Similarity Analysis

In [ ]:
from pathlib import Path

import pandas as pd

from src.Similarity.LoadData import SimilarityDataLoader
from src.Similarity.TemporalAggregation import SimilarityTemporalAggregator
from src.Similarity.Preprocessing import SimilarityPreprocessor
from src.Similarity.FeatureAnalysis import SimilarityFeatureAnalysis
from src.Similarity.SimilarityAnalyzer import SimilarityAnalyzer
from src.Plots.SimilarityDistanceHeatmapPlot import SimilarityDistanceHeatmapPlot
from src.Plots.SimilarityDendrogramPlot import SimilarityDendrogramPlot

## 1. Create Temporal Cuts

Seleciona um periodo de tempo em minutos a partir do início de cada ataque, preservando todos os fluxos existentes no intervalo. Nesta etapa são criados `FlowStart` e `FlowEnd`.

In [ ]:
experimentName = "CICDDoS2019_01-12_Attack_Similarity"
dataDirectory = Path(r"E:\CICDDoS2019\01-12\CSV")
outputRoot = Path("output/Similarity") / experimentName

attackFiles = {
    "NTP": "DrDoS_NTP.csv",
    "DNS": "DrDoS_DNS.csv",
    "LDAP": "DrDoS_LDAP.csv",
    "MSSQL": "DrDoS_MSSQL.csv",
    "NETBIOS": "DrDoS_NetBIOS.csv",
    "SNMP": "DrDoS_SNMP.csv",
    "SSDP": "DrDoS_SSDP.csv",
    "UDP": "DrDoS_UDP.csv",
    "UDP-LAG": "UDPLag.csv",
    "SYN": "Syn.csv",
    "TFTP": "TFTP.csv",
}

durationMinutes = 5.0
chunkSize = 100_000
normalLabels = ["BENIGN", "NORMAL", "0"]

cutsDirectory = outputRoot / "cuts"
cutSummaryPath = outputRoot / "reports" / "temporal_cuts.csv"

loader = SimilarityDataLoader(
    normalLabels=normalLabels,
    chunkSize=chunkSize,
)

loader.createCuts(
    dataDirectory=dataDirectory,
    attackFiles=attackFiles,
    durationMinutes=durationMinutes,
    outputDirectory=cutsDirectory,
    summaryPath=cutSummaryPath,
)

display(pd.read_csv(cutSummaryPath))

,Attack,SourceFile,CutStart,CutEnd,DurationMinutes,FlowCount,AttackFlowCount,BenignFlowCount,OtherAttackFlowCount
0,NTP,E:\CICDDoS2019\01-12\CSV\DrDoS_NTP.csv,2018-01-12 09:17:16.404686,2018-01-12 09:22:16.404686,5.0,1499,329,1170,0
1,DNS,E:\CICDDoS2019\01-12\CSV\DrDoS_DNS.csv,2018-01-12 10:51:39.820844,2018-01-12 10:56:39.820844,5.0,7200,6566,634,0
2,LDAP,E:\CICDDoS2019\01-12\CSV\DrDoS_LDAP.csv,2018-01-12 11:22:40.255362,2018-01-12 11:27:40.255362,5.0,1135343,1134487,856,0
3,MSSQL,E:\CICDDoS2019\01-12\CSV\DrDoS_MSSQL.csv,2018-01-12 11:32:32.915443,2018-01-12 11:37:32.915443,5.0,167697,167044,653,0
4,NETBIOS,E:\CICDDoS2019\01-12\CSV\DrDoS_NetBIOS.csv,2018-01-12 11:47:08.463790,2018-01-12 11:52:08.463790,5.0,652530,651738,792,0
5,SNMP,E:\CICDDoS2019\01-12\CSV\DrDoS_SNMP.csv,2018-01-12 12:00:13.902784,2018-01-12 12:05:13.902784,5.0,231792,231387,405,0
6,SSDP,E:\CICDDoS2019\01-12\CSV\DrDoS_SSDP.csv,2018-01-12 12:23:13.663477,2018-01-12 12:28:13.663477,5.0,552699,552566,133,0
7,UDP,E:\CICDDoS2019\01-12\CSV\DrDoS_UDP.csv,2018-01-12 12:36:57.628218,2018-01-12 12:41:57.628218,5.0,66456,66281,175,0
8,UDP-LAG,E:\CICDDoS2019\01-12\CSV\UDPLag.csv,2018-01-12 13:04:45.928674,2018-01-12 13:09:45.928674,5.0,44346,43797,483,66
9,SYN,E:\CICDDoS2019\01-12\CSV\Syn.csv,2018-01-12 13:30:30.741566,2018-01-12 13:35:30.741566,5.0,1582681,1582289,392,0


## 2. Aggregate

As duas representações são construídas a partir dos mesmos arquivos em `cuts/`. Janelas sem qualquer fluxo encerrado recebem `Label = NOLABEL`; `BENIGN` fica reservado a janelas que realmente contêm fluxo benigno.

### 2.1 Attack-Only

Filtra cada recorte para manter somente os fluxos do ataque antes da agregação.

In [ ]:
experimentName = "CICDDoS2019_01-12_Attack_Similarity"
outputRoot = Path("output/Similarity") / experimentName

cutsDirectory = outputRoot / "cuts"
cutSummaryPath = outputRoot / "reports" / "temporal_cuts.csv"
perAttackDirectory = outputRoot / "windows" / "attack_only"
combinedOutputPath = outputRoot / "datasets" / "attack_only_raw_0_5s.csv"
summaryOutputPath = outputRoot / "reports" / "attack_only_window_summary.csv"

windowSeconds = 0.5
fillEmptyWindows = True
normalLabels = ["BENIGN", "NORMAL", "0"]
noFlowLabel = "NOLABEL"

aggregator = SimilarityTemporalAggregator(
    windowSeconds=windowSeconds,
    normalLabels=normalLabels,
    fillEmptyWindows=fillEmptyWindows,
    noFlowLabel=noFlowLabel,
)

aggregator.aggregateDirectory(
    cutsDirectory=cutsDirectory,
    cutSummaryPath=cutSummaryPath,
    mode="attack_only",
    perAttackDirectory=perAttackDirectory,
    combinedOutputPath=combinedOutputPath,
    summaryOutputPath=summaryOutputPath,
)

print("Dataset bruto Attack-Only:", combinedOutputPath)
display(pd.read_csv(summaryOutputPath))

Dataset bruto Attack-Only: output\Similarity\CICDDoS2019_01-12_Attack_Similarity\datasets\attack_only_raw_0_5s.csv


,Attack,Mode,WindowCount,AttackWindows,BenignWindows,NoFlowWindows,TotalFlows,AttackFlows,BenignFlows,OtherAttackFlows,MeanAttackRatio,StdAttackRatio
0,NTP,attack_only,600,114,0,486,329,329,0,0,1.0,0.0
1,DNS,attack_only,600,521,0,79,6566,6566,0,0,1.0,0.0
2,LDAP,attack_only,600,596,0,4,1134487,1134487,0,0,1.0,0.0
3,MSSQL,attack_only,600,129,0,471,167044,167044,0,0,1.0,0.0
4,NETBIOS,attack_only,600,380,0,220,651738,651738,0,0,1.0,0.0
5,SNMP,attack_only,600,185,0,415,231387,231387,0,0,1.0,0.0
6,SSDP,attack_only,600,566,0,34,552566,552566,0,0,1.0,0.0
7,UDP,attack_only,600,405,0,195,66281,66281,0,0,1.0,0.0
8,UDP-LAG,attack_only,600,352,0,248,43797,43797,0,0,1.0,0.0
9,SYN,attack_only,600,474,0,126,1582289,1582289,0,0,1.0,0.0


### 2.2 Mixed Traffic

Mantém ataque e tráfego benigno do mesmo recorte temporal antes da agregação.

In [ ]:
experimentName = "CICDDoS2019_01-12_Attack_Similarity"
outputRoot = Path("output/Similarity") / experimentName

cutsDirectory = outputRoot / "cuts"
cutSummaryPath = outputRoot / "reports" / "temporal_cuts.csv"
perAttackDirectory = outputRoot / "windows" / "mixed_traffic"
combinedOutputPath = outputRoot / "datasets" / "mixed_traffic_raw_0_5s.csv"
summaryOutputPath = outputRoot / "reports" / "mixed_traffic_window_summary.csv"

windowSeconds = 0.5
fillEmptyWindows = True
normalLabels = ["BENIGN", "NORMAL", "0"]
noFlowLabel = "NOLABEL"

aggregator = SimilarityTemporalAggregator(
    windowSeconds=windowSeconds,
    normalLabels=normalLabels,
    fillEmptyWindows=fillEmptyWindows,
    noFlowLabel=noFlowLabel,
)

aggregator.aggregateDirectory(
    cutsDirectory=cutsDirectory,
    cutSummaryPath=cutSummaryPath,
    mode="mixed_traffic",
    perAttackDirectory=perAttackDirectory,
    combinedOutputPath=combinedOutputPath,
    summaryOutputPath=summaryOutputPath,
)

print("Dataset bruto Mixed Traffic:", combinedOutputPath)
display(pd.read_csv(summaryOutputPath))

Dataset bruto Mixed Traffic: output\Similarity\CICDDoS2019_01-12_Attack_Similarity\datasets\mixed_traffic_raw_0_5s.csv


,Attack,Mode,WindowCount,AttackWindows,BenignWindows,NoFlowWindows,TotalFlows,AttackFlows,BenignFlows,OtherAttackFlows,MeanAttackRatio,StdAttackRatio
0,NTP,mixed_traffic,600,114,152,334,1499,329,1170,0,0.361194,0.452954
1,DNS,mixed_traffic,600,521,9,70,7200,6566,634,0,0.931866,0.197912
2,LDAP,mixed_traffic,600,596,0,4,1135343,1134487,856,0,0.999254,0.002263
3,MSSQL,mixed_traffic,600,129,94,377,167697,167044,653,0,0.548468,0.487166
4,NETBIOS,mixed_traffic,600,380,34,186,652530,651738,792,0,0.871027,0.310589
5,SNMP,mixed_traffic,600,185,51,364,231792,231387,405,0,0.738560,0.418118
6,SSDP,mixed_traffic,600,566,3,31,552699,552566,133,0,0.987495,0.089061
7,UDP,mixed_traffic,600,405,8,187,66456,66281,175,0,0.947557,0.196460
8,UDP-LAG,mixed_traffic,600,352,50,198,44346,43797,483,66,0.810921,0.359452
9,SYN,mixed_traffic,600,474,0,126,1582681,1582289,392,0,0.999751,0.000799


## 3. Preprocessing

Cada subseção reabre diretamente o CSV bruto indicado. `BENIGN` e `NOLABEL` não entram na análise estatística; as janelas de ataque são balanceadas por random undersampling, valores inválidos são imputados pela mediana do próprio ataque e a normalização é feita em lote com um único StandardScaler.

### 3.1 Attack-Only

In [ ]:
experimentName = "CICDDoS2019_01-12_Attack_Similarity"
outputRoot = Path("output/Similarity") / experimentName

datasetPath = outputRoot / "datasets" / "attack_only_raw_0_5s.csv"
analysisOutput = outputRoot / "analysis" / "attack_only"

selectedFeatures = None
nearConstantRatio = 0.999
correlationThreshold = 0.995
missingDisplayThreshold = 51.0
randomSeed = 42
noFlowLabel = "NOLABEL"

preprocessor = SimilarityPreprocessor(
    nearConstantRatio=nearConstantRatio,
    correlationThreshold=correlationThreshold,
    randomSeed=randomSeed,
    missingDisplayThreshold=missingDisplayThreshold,
    noFlowLabel=noFlowLabel,
)

result = preprocessor.runFromFile(
    datasetPath=datasetPath,
    outputDirectory=analysisOutput,
    selectedFeatures=selectedFeatures,
)

print("Base balanceada:", result["balancedDatasetPath"])
print("Base padronizada:", result["standardizedDatasetPath"])
print("Quantidade de features:", len(result["features"]))

print("\nBalanceamento aplicado:")
display(pd.DataFrame.from_dict(result["balanceSummary"], orient="index"))

Base balanceada: output\Similarity\CICDDoS2019_01-12_Attack_Similarity\analysis\attack_only\datasets\balanced_attack_windows.csv
Base padronizada: output\Similarity\CICDDoS2019_01-12_Attack_Similarity\analysis\attack_only\datasets\standardized_attack_windows.csv
Quantidade de features: 58

Balanceamento aplicado:


,availableWindows,selectedWindows,droppedWindows
DNS,521,114,407
LDAP,596,114,482
MSSQL,129,114,15
NETBIOS,380,114,266
NTP,114,114,0
SNMP,185,114,71
SSDP,566,114,452
SYN,474,114,360
TFTP,172,114,58
UDP,405,114,291


#### 3.1.1 Missing Values

In [8]:
print(f"\nFeatures com pelo menos {missingDisplayThreshold:.1f}% de valores ausentes em algum ataque:")
if result["missingSummary"].empty:
    print("Nenhuma feature ultrapassou o limiar.")
else:
    display(result["missingSummary"].sort_values(["MissingPercentage", "Attack"], ascending=[False, True]))


Features com pelo menos 51.0% de valores ausentes em algum ataque:


,Attack,Feature,MissingCount,TotalCount,MissingPercentage
35,UDP-LAG,Bwd_Packet_Length_Mean,102,114,89.473684
36,UDP-LAG,Bwd_Packet_Length_Std,102,114,89.473684
37,UDP-LAG,Bwd_IAT_Mean,102,114,89.473684
38,UDP-LAG,Bwd_IAT_Std,102,114,89.473684
39,UDP-LAG,Avg_Bwd_Segment_Size,102,114,89.473684
30,UDP,Bwd_Packet_Length_Mean,101,114,88.596491
31,UDP,Bwd_Packet_Length_Std,101,114,88.596491
32,UDP,Bwd_IAT_Mean,101,114,88.596491
33,UDP,Bwd_IAT_Std,101,114,88.596491
34,UDP,Avg_Bwd_Segment_Size,101,114,88.596491


### 3.2 Mixed Traffic

In [ ]:
experimentName = "CICDDoS2019_01-12_Attack_Similarity"
outputRoot = Path("output/Similarity") / experimentName

datasetPath = outputRoot / "datasets" / "mixed_traffic_raw_0_5s.csv"
analysisOutput = outputRoot / "analysis" / "mixed_traffic"

selectedFeatures = None
nearConstantRatio = 0.999
correlationThreshold = 0.995
missingDisplayThreshold = 30.0
randomSeed = 42
noFlowLabel = "NOLABEL"

preprocessor = SimilarityPreprocessor(
    nearConstantRatio=nearConstantRatio,
    correlationThreshold=correlationThreshold,
    randomSeed=randomSeed,
    missingDisplayThreshold=missingDisplayThreshold,
    noFlowLabel=noFlowLabel,
)

result = preprocessor.runFromFile(
    datasetPath=datasetPath,
    outputDirectory=analysisOutput,
    selectedFeatures=selectedFeatures,
)

print("Base balanceada:", result["balancedDatasetPath"])
print("Base padronizada:", result["standardizedDatasetPath"])
print("Quantidade de features:", len(result["features"]))

print("\nBalanceamento aplicado:")
display(pd.DataFrame.from_dict(result["balanceSummary"], orient="index"))

Base balanceada: output\Similarity\CICDDoS2019_01-12_Attack_Similarity\analysis\mixed_traffic\datasets\balanced_attack_windows.csv
Base padronizada: output\Similarity\CICDDoS2019_01-12_Attack_Similarity\analysis\mixed_traffic\datasets\standardized_attack_windows.csv
Quantidade de features: 62

Balanceamento aplicado:


,availableWindows,selectedWindows,droppedWindows
DNS,521,114,407
LDAP,596,114,482
MSSQL,129,114,15
NETBIOS,380,114,266
NTP,114,114,0
SNMP,185,114,71
SSDP,566,114,452
SYN,474,114,360
TFTP,172,114,58
UDP,405,114,291


#### 3.2.1 Missing Values

In [ ]:
# print(f"\nFeatures com pelo menos {missingDisplayThreshold:.1f}% de valores ausentes em algum ataque:")
# if result["missingSummary"].empty:
#     print("Nenhuma feature ultrapassou o limiar.")
# else:
#     display(result["missingSummary"].sort_values(["MissingPercentage", "Attack"], ascending=[False, True]))

## 4. Statistical Characterization

As estatísticas são calculadas a partir dos arquivos imputados salvos em disco e usam somente as features registradas em `feature_selection.json`.

### 4.1 Attack-Only

In [ ]:
experimentName = "CICDDoS2019_01-12_Attack_Similarity"
outputRoot = Path("output/Similarity") / experimentName
analysisOutput = outputRoot / "analysis" / "attack_only"

datasetPath = analysisOutput / "datasets" / "imputed_attack_windows.csv"
featureSelectionPath = analysisOutput / "reports" / "feature_selection.json"
statisticsPath = analysisOutput / "reports" / "attack_feature_statistics.csv"

SimilarityFeatureAnalysis().characterizeFromFile(
    datasetPath=datasetPath,
    featureSelectionPath=featureSelectionPath,
    outputPath=statisticsPath,
)

display(pd.read_csv(statisticsPath).head(30))

,Attack,Feature,Count,Mean,Median,Std,Q1,Q3,Minimum,Maximum
0,DNS,FlowCount,114,1.256140e+01,1.000000e+01,8.203588e+00,7.000000,1.700000e+01,1.000000,3.600000e+01
1,DNS,UniqueSourceIPs,114,1.614035e+00,2.000000e+00,4.868223e-01,1.000000,2.000000e+00,1.000000,2.000000e+00
2,DNS,UniqueSourcePorts,114,3.122807e+00,3.000000e+00,1.727692e+00,2.000000,4.000000e+00,1.000000,8.000000e+00
3,DNS,UniqueDestinationPorts,114,1.192982e+01,1.000000e+01,8.248040e+00,6.000000,1.700000e+01,1.000000,3.500000e+01
4,DNS,UniqueProtocols,114,1.157895e+00,1.000000e+00,3.646423e-01,1.000000,1.000000e+00,1.000000,2.000000e+00
5,DNS,SourceIPEntropy,114,3.172730e-01,3.095434e-01,3.074818e-01,0.000000,5.435644e-01,-0.000000,1.000000e+00
6,DNS,SourcePortEntropy,114,9.373078e-01,9.445302e-01,7.041435e-01,0.468996,1.348683e+00,-0.000000,2.663533e+00
7,DNS,DestinationPortEntropy,114,3.106297e+00,3.321928e+00,1.282132e+00,2.521641,4.058814e+00,-0.000000,5.114369e+00
8,DNS,ProtocolEntropy,114,8.839115e-02,0.000000e+00,2.319169e-01,0.000000,-0.000000e+00,-0.000000,1.000000e+00
9,DNS,Flow_Duration,114,5.157683e+05,5.088087e+04,2.845127e+06,31064.000000,6.420265e+04,1.000000,2.670281e+07


### 4.2 Mixed Traffic

In [ ]:
experimentName = "CICDDoS2019_01-12_Attack_Similarity"
outputRoot = Path("output/Similarity") / experimentName
analysisOutput = outputRoot / "analysis" / "mixed_traffic"

datasetPath = analysisOutput / "datasets" / "imputed_attack_windows.csv"
featureSelectionPath = analysisOutput / "reports" / "feature_selection.json"
statisticsPath = analysisOutput / "reports" / "attack_feature_statistics.csv"

SimilarityFeatureAnalysis().characterizeFromFile(
    datasetPath=datasetPath,
    featureSelectionPath=featureSelectionPath,
    outputPath=statisticsPath,
)

display(pd.read_csv(statisticsPath).head(30))

,Attack,Feature,Count,Mean,Median,Std,Q1,Q3,Minimum,Maximum
0,DNS,FlowCount,114,1.352632e+01,1.100000e+01,8.498927e+00,7.250000,1.800000e+01,1.000000,3.900000e+01
1,DNS,UniqueSourceIPs,114,1.903509e+00,2.000000e+00,9.997307e-01,1.000000,2.000000e+00,1.000000,7.000000e+00
2,DNS,UniqueDestinationIPs,114,2.070175e+00,2.000000e+00,1.443189e+00,1.000000,2.000000e+00,1.000000,1.000000e+01
3,DNS,UniqueSourcePorts,114,3.719298e+00,3.000000e+00,2.280769e+00,2.000000,5.000000e+00,1.000000,1.200000e+01
4,DNS,UniqueDestinationPorts,114,1.227193e+01,1.000000e+01,8.288954e+00,6.250000,1.700000e+01,1.000000,3.600000e+01
5,DNS,UniqueProtocols,114,1.280702e+00,1.000000e+00,4.684572e-01,1.000000,2.000000e+00,1.000000,3.000000e+00
6,DNS,SourceIPEntropy,114,4.471083e-01,3.372901e-01,4.858891e-01,0.000000,7.056380e-01,-0.000000,2.401058e+00
7,DNS,DestinationIPEntropy,114,4.949781e-01,3.372901e-01,6.083244e-01,0.000000,7.219281e-01,-0.000000,3.180833e+00
8,DNS,SourcePortEntropy,114,1.123714e+00,1.026663e+00,8.263657e-01,0.498677,1.547425e+00,-0.000000,3.546594e+00
9,DNS,DestinationPortEntropy,114,3.129898e+00,3.321928e+00,1.273535e+00,2.521641,4.058814e+00,-0.000000,5.129283e+00


## 5. Mahalanobis Analysis

### 5.1 Attack-Only

#### 5.1.1 Matrix

In [ ]:
experimentName = "CICDDoS2019_01-12_Attack_Similarity"
outputRoot = Path("output/Similarity") / experimentName
analysisOutput = outputRoot / "analysis" / "attack_only"

standardizedDatasetPath = analysisOutput / "datasets" / "standardized_attack_windows.csv"
featureSelectionPath = analysisOutput / "reports" / "feature_selection.json"
reportDirectory = analysisOutput / "reports"

result = SimilarityAnalyzer().runFromFile(
    standardizedDatasetPath=standardizedDatasetPath,
    featureSelectionPath=featureSelectionPath,
    reportDirectory=reportDirectory,
)

display(pd.read_csv(result["distanceMatrixPath"], index_col=0))

,DNS,LDAP,MSSQL,NETBIOS,NTP,SNMP,SSDP,SYN,TFTP,UDP,UDP-LAG
DNS,0.000000,6.649202,4.298657,4.720471,5.519392,4.710910,3.100072,7.897245,4.293319,2.667256,3.045900
LDAP,6.649202,0.000000,5.912226,7.215878,7.899172,7.384311,5.887860,9.636104,7.179685,6.314121,6.205834
MSSQL,4.298657,5.912226,0.000000,3.142563,3.827795,3.028345,3.758058,7.797878,4.098488,3.680481,3.498144
NETBIOS,4.720471,7.215878,3.142563,0.000000,4.517411,1.595338,4.804296,7.481939,3.604479,4.218433,4.070093
NTP,5.519392,7.899172,3.827795,4.517411,0.000000,3.619838,5.623811,7.979689,4.533778,5.027703,4.977100
SNMP,4.710910,7.384311,3.028345,1.595338,3.619838,0.000000,4.936081,7.492800,3.495517,4.193444,4.104314
SSDP,3.100072,5.887860,3.758058,4.804296,5.623811,4.936081,0.000000,7.674079,4.235150,2.114924,2.178729
SYN,7.897245,9.636104,7.797878,7.481939,7.979689,7.492800,7.674079,0.000000,5.717745,7.095695,7.154430
TFTP,4.293319,7.179685,4.098488,3.604479,4.533778,3.495517,4.235150,5.717745,0.000000,3.183026,3.135667
UDP,2.667256,6.314121,3.680481,4.218433,5.027703,4.193444,2.114924,7.095695,3.183026,0.000000,1.166778


#### 5.1.2 Distance Levels

In [ ]:
experimentName = "CICDDoS2019_01-12_Attack_Similarity"
outputRoot = Path("output/Similarity") / experimentName
analysisOutput = outputRoot / "analysis" / "attack_only"

pairwisePath = analysisOutput / "reports" / "pairwise_mahalanobis_distances.csv"
levelsPath = analysisOutput / "reports" / "pairwise_distance_levels.csv"

similarQuantile = 0.33
distinctQuantile = 0.67

levelsPath, similarLimit, distinctLimit = SimilarityAnalyzer().classifyDistanceFile(
    pairwisePath=pairwisePath,
    outputPath=levelsPath,
    similarQuantile=similarQuantile,
    distinctQuantile=distinctQuantile,
)

levels = pd.read_csv(levelsPath)
print("Pares semelhantes:")
display(levels.query("DistanceLevel == 'similar'"))

print("Pares intermediários:")
display(levels.query("DistanceLevel == 'intermediate'"))

print("Pares distintos:")
display(levels.query("DistanceLevel == 'distinct'").sort_values("MahalanobisDistance", ascending=False))

Pares semelhantes:


,AttackA,AttackB,MahalanobisDistance,DistanceLevel
0,UDP,UDP-LAG,1.166778,similar
1,NETBIOS,SNMP,1.595338,similar
2,SSDP,UDP,2.114924,similar
3,SSDP,UDP-LAG,2.178729,similar
4,DNS,UDP,2.667256,similar
5,MSSQL,SNMP,3.028345,similar
6,DNS,UDP-LAG,3.045900,similar
7,DNS,SSDP,3.100072,similar
8,TFTP,UDP-LAG,3.135667,similar
9,MSSQL,NETBIOS,3.142563,similar


Pares intermediários:


,AttackA,AttackB,MahalanobisDistance,DistanceLevel
18,NETBIOS,UDP-LAG,4.070093,intermediate
19,MSSQL,TFTP,4.098488,intermediate
20,SNMP,UDP-LAG,4.104314,intermediate
21,SNMP,UDP,4.193444,intermediate
22,NETBIOS,UDP,4.218433,intermediate
23,SSDP,TFTP,4.235150,intermediate
24,DNS,TFTP,4.293319,intermediate
25,DNS,MSSQL,4.298657,intermediate
26,NETBIOS,NTP,4.517411,intermediate
27,NTP,TFTP,4.533778,intermediate


Pares distintos:


,AttackA,AttackB,MahalanobisDistance,DistanceLevel
54,LDAP,SYN,9.636104,distinct
53,NTP,SYN,7.979689,distinct
52,LDAP,NTP,7.899172,distinct
51,DNS,SYN,7.897245,distinct
50,MSSQL,SYN,7.797878,distinct
49,SSDP,SYN,7.674079,distinct
48,SNMP,SYN,7.492800,distinct
47,NETBIOS,SYN,7.481939,distinct
46,LDAP,SNMP,7.384311,distinct
45,LDAP,NETBIOS,7.215878,distinct


#### 5.1.3 Heatmap and Dendrogram

In [21]:
experimentName = "CICDDoS2019_01-12_Attack_Similarity"
analysisOutput = Path("output/Similarity") / experimentName / "analysis" / "attack_only"

distanceMatrixPath = analysisOutput / "reports" / "mahalanobis_distance_matrix.csv"
outputPath = analysisOutput / "plots" / "attack_distance_dendrogram.png"

SimilarityDendrogramPlot().plot(
    pd.read_csv(distanceMatrixPath, index_col=0),
    outputPath,
)

print("Dendrograma salvo em:", outputPath)

Dendrograma salvo em: output\Similarity\CICDDoS2019_01-12_Attack_Similarity\analysis\attack_only\plots\attack_distance_dendrogram.png


In [22]:
experimentName = "CICDDoS2019_01-12_Attack_Similarity"
analysisOutput = Path("output/Similarity") / experimentName / "analysis" / "attack_only"

distanceMatrixPath = analysisOutput / "reports" / "mahalanobis_distance_matrix.csv"
outputPath = analysisOutput / "plots" / "mahalanobis_distance_heatmap.png"

SimilarityDistanceHeatmapPlot().plot(
    pd.read_csv(distanceMatrixPath, index_col=0),
    outputPath,
)

print("Heatmap salvo em:", outputPath)

Heatmap salvo em: output\Similarity\CICDDoS2019_01-12_Attack_Similarity\analysis\attack_only\plots\mahalanobis_distance_heatmap.png


### 5.2 Mixed Traffic

#### 5.2.1 Matrix

In [17]:
experimentName = "CICDDoS2019_01-12_Attack_Similarity"
outputRoot = Path("output/Similarity") / experimentName
analysisOutput = outputRoot / "analysis" / "mixed_traffic"

standardizedDatasetPath = analysisOutput / "datasets" / "standardized_attack_windows.csv"
featureSelectionPath = analysisOutput / "reports" / "feature_selection.json"
reportDirectory = analysisOutput / "reports"

result = SimilarityAnalyzer().runFromFile(
    standardizedDatasetPath=standardizedDatasetPath,
    featureSelectionPath=featureSelectionPath,
    reportDirectory=reportDirectory,
)

display(pd.read_csv(result["distanceMatrixPath"], index_col=0))

,DNS,LDAP,MSSQL,NETBIOS,NTP,SNMP,SSDP,SYN,TFTP,UDP,UDP-LAG
DNS,0.000000,7.179860,4.054234,4.015774,4.723114,4.222182,3.214675,7.592730,4.045153,2.610019,2.916491
LDAP,7.179860,0.000000,6.091738,7.358910,7.878876,7.552632,6.407956,9.922407,7.602991,6.770887,6.768215
MSSQL,4.054234,6.091738,0.000000,3.093654,3.463996,3.021972,3.446218,7.434165,3.866711,3.205026,3.008381
NETBIOS,4.015774,7.358910,3.093654,0.000000,3.834469,1.661412,4.363290,7.031970,3.182177,3.538692,3.385128
NTP,4.723114,7.878876,3.463996,3.834469,0.000000,3.061403,5.004598,7.406885,3.799671,4.153922,4.037673
SNMP,4.222182,7.552632,3.021972,1.661412,3.061403,0.000000,4.486191,7.020508,3.013012,3.569820,3.428725
SSDP,3.214675,6.407956,3.446218,4.363290,5.004598,4.486191,0.000000,7.451095,4.176638,2.140271,2.268810
SYN,7.592730,9.922407,7.434165,7.031970,7.406885,7.020508,7.451095,0.000000,5.398964,6.818025,6.934797
TFTP,4.045153,7.602991,3.866711,3.182177,3.799671,3.013012,4.176638,5.398964,0.000000,3.043039,3.068627
UDP,2.610019,6.770887,3.205026,3.538692,4.153922,3.569820,2.140271,6.818025,3.043039,0.000000,1.152419


#### 5.2.2 Distance Levels

In [18]:
experimentName = "CICDDoS2019_01-12_Attack_Similarity"
outputRoot = Path("output/Similarity") / experimentName
analysisOutput = outputRoot / "analysis" / "mixed_traffic"

pairwisePath = analysisOutput / "reports" / "pairwise_mahalanobis_distances.csv"
levelsPath = analysisOutput / "reports" / "pairwise_distance_levels.csv"

similarQuantile = 0.33
distinctQuantile = 0.67

levelsPath, similarLimit, distinctLimit = SimilarityAnalyzer().classifyDistanceFile(
    pairwisePath=pairwisePath,
    outputPath=levelsPath,
    similarQuantile=similarQuantile,
    distinctQuantile=distinctQuantile,
)

levels = pd.read_csv(levelsPath)
print("Pares semelhantes:")
display(levels.query("DistanceLevel == 'similar'"))

print("Pares intermediários:")
display(levels.query("DistanceLevel == 'intermediate'"))

print("Pares distintos:")
display(levels.query("DistanceLevel == 'distinct'").sort_values("MahalanobisDistance", ascending=False))

Pares semelhantes:


,AttackA,AttackB,MahalanobisDistance,DistanceLevel
0,UDP,UDP-LAG,1.152419,similar
1,NETBIOS,SNMP,1.661412,similar
2,SSDP,UDP,2.140271,similar
3,SSDP,UDP-LAG,2.268810,similar
4,DNS,UDP,2.610019,similar
5,DNS,UDP-LAG,2.916491,similar
6,MSSQL,UDP-LAG,3.008381,similar
7,SNMP,TFTP,3.013012,similar
8,MSSQL,SNMP,3.021972,similar
9,TFTP,UDP,3.043039,similar


Pares intermediários:


,AttackA,AttackB,MahalanobisDistance,DistanceLevel
18,MSSQL,SSDP,3.446218,intermediate
19,MSSQL,NTP,3.463996,intermediate
20,NETBIOS,UDP,3.538692,intermediate
21,SNMP,UDP,3.569820,intermediate
22,NTP,TFTP,3.799671,intermediate
23,NETBIOS,NTP,3.834469,intermediate
24,MSSQL,TFTP,3.866711,intermediate
25,DNS,NETBIOS,4.015774,intermediate
26,NTP,UDP-LAG,4.037673,intermediate
27,DNS,TFTP,4.045153,intermediate


Pares distintos:


,AttackA,AttackB,MahalanobisDistance,DistanceLevel
54,LDAP,SYN,9.922407,distinct
53,LDAP,NTP,7.878876,distinct
52,LDAP,TFTP,7.602991,distinct
51,DNS,SYN,7.592730,distinct
50,LDAP,SNMP,7.552632,distinct
49,SSDP,SYN,7.451095,distinct
48,MSSQL,SYN,7.434165,distinct
47,NTP,SYN,7.406885,distinct
46,LDAP,NETBIOS,7.358910,distinct
45,DNS,LDAP,7.179860,distinct


#### 5.2.3 Heatmap and Dendrogram

In [23]:
experimentName = "CICDDoS2019_01-12_Attack_Similarity"
analysisOutput = Path("output/Similarity") / experimentName / "analysis" / "mixed_traffic"

distanceMatrixPath = analysisOutput / "reports" / "mahalanobis_distance_matrix.csv"
outputPath = analysisOutput / "plots" / "mahalanobis_distance_heatmap.png"

SimilarityDistanceHeatmapPlot().plot(
    pd.read_csv(distanceMatrixPath, index_col=0),
    outputPath,
)

print("Heatmap salvo em:", outputPath)

Heatmap salvo em: output\Similarity\CICDDoS2019_01-12_Attack_Similarity\analysis\mixed_traffic\plots\mahalanobis_distance_heatmap.png


In [24]:
experimentName = "CICDDoS2019_01-12_Attack_Similarity"
analysisOutput = Path("output/Similarity") / experimentName / "analysis" / "mixed_traffic"

distanceMatrixPath = analysisOutput / "reports" / "mahalanobis_distance_matrix.csv"
outputPath = analysisOutput / "plots" / "attack_distance_dendrogram.png"

SimilarityDendrogramPlot().plot(
    pd.read_csv(distanceMatrixPath, index_col=0),
    outputPath,
)

print("Dendrograma salvo em:", outputPath)

Dendrograma salvo em: output\Similarity\CICDDoS2019_01-12_Attack_Similarity\analysis\mixed_traffic\plots\attack_distance_dendrogram.png
